# Exploring Training Sizes

The goal of this experiment is to undestrand how the training size affects the performance of the model. We will train the model in the following number of samples by model: (2k, 3k, 5k, 10k, 15k and 20k). We will use the same collections for all models but with random choice.

In [6]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch
import types
import sys
import os
import json
import joblib
import random
from pathlib import Path


from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from numpy import float64
from sklearn.metrics import roc_auc_score
from utils.metrics.calculate_metric import calculate_agg_metric
from utils.set_random_seed import set_random_seed
from interpret.glassbox import ExplainableBoostingClassifier
from utils.model_loaders import load_logistic_models_for_subfolder



set_random_seed(42)
EXPERIMENTS = ["experiment_1", "experiment_4"]

In [ ]:
def _get_split_sets(df, subfolder: str, split: str, num_collections: int):
    TOTAL_NUM_COLLECTIONS = 20000
    
    collections = random.sample(range(TOTAL_NUM_COLLECTIONS), num_collections, random_state=42)
    split_df = df.filter((pl.col("subfolder") == subfolder) & (pl.col("split") == split)).filter(pl.col("collection_idx").is_in(collections))

    X_split = []
    y_split = []

    for exp in EXPERIMENTS:
        exp_df = split_df.filter(pl.col("experiment") == exp)

        X_exp = exp_df.select("input").to_numpy()
        X_exp = np.array([i[0] for i in X_exp])
        y_exp = exp_df.select("evaluation").to_numpy()

        # Data is interleaved: first 500 elements go to position 0 of each array,
        # next 500 elements go to position 1 of each array, etc.
        num_arrays = 500

        X_exp_reshaped = []
        y_exp_reshaped = []
        for i in range(num_arrays):
            # Take every 500th element starting from index i
            X_exp_reshaped.append(X_exp[i::num_arrays])
            y_exp_reshaped.append(y_exp[i::num_arrays])

        X_split.append(X_exp_reshaped)
        y_split.append(y_exp_reshaped)

    return X_split, y_split


def get_train_sets(df, subfolder):
    return _get_split_sets(df, subfolder=subfolder, split="train")


def get_test_sets(df, subfolder):
    return _get_split_sets(df, subfolder=subfolder, split="test")

In [9]:
df = []

for subfolder in ["voting_alt1"]:
    for exp in EXPERIMENTS:
        for split in ["train"]:
            try:
                load_df = pl.read_ipc(f"../binary_collections/{subfolder}/{exp}/{split}.feather")
                load_df = load_df.with_columns([
                    pl.lit(subfolder).alias("subfolder"),
                    pl.lit(exp).alias("experiment"),
                    pl.lit(split).alias("split"),
                ])
                df.append(load_df)
            except Exception as e:
                print(f"Could not load {subfolder}/{exp}/{split}: {e}")
df = pl.concat(df)
df.head()

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


collection_idx,test_idx,input,evaluation,subfolder,experiment,split
i64,i64,"array[i64, 100]",i32,str,str,str
0,0,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,1,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,2,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
0,3,"[0, 1, … 0]",0,"""voting_alt1""","""experiment_1""","""train"""
0,4,"[0, 1, … 0]",1,"""voting_alt1""","""experiment_1""","""train"""
